# <span style="color:#0275D8">Tokenization & Embeddings</span>

Before any recurrent neural network can process text, the raw strings must undergo a deterministic transformation pipeline to convert human language into continuous numerical coordinates.

---

## <span style="color:#20639B">What is Tokenization?</span>

Computers cannot process letters, words, or sentences directly; they operate strictly on numbers. **Tokenization** is the structural preprocessing phase that partitions a raw string of text into smaller, discrete units called **tokens**. Depending on the system architecture, tokens can represent:
* **Whole words** (Word-level)
* **Subword components** (Morpheme-level)
* **Single characters** (Character-level)

Once partitioned, each unique token is mapped to a static integer identifier based on a pre-compiled index library known as the **Vocabulary (Vocab)**.

---

## <span style="color:#3CAEA3">What are Embeddings?</span>

Feeding raw scalar token IDs (passing `102` or `4502` directly) into a neural network creates a fatal structural flaw: the model treats them as arbitrary categorical data. The network cannot natively deduce that ID `102` (*"cat"*) shares a closer semantic relationship to ID `103` (*"dog"*) than it does to ID `8943` (*"airplane"*).

An **Embedding Layer** maps each discrete token ID to a continuous, high-dimensional **dense vector** (a array of floating-point numbers). This vector serves as a coordinate point within a high-dimensional geometric workspace. In this space, the **geometric distance** (Cosine or Euclidean distance) between two vectors directly correlates to their **semantic similarity**.

---

## <span style="color:#ED553B">The Math Behind Embeddings</span>

Mathematically, an embedding layer is structured as a large weight matrix (a parameterized lookup table) denoted as <span style="color:#ED553B">$\mathbf{W_e}$</span>.

Given a Vocabulary size $V$ and a hidden vector dimensionality $D$, the matrix space is defined as:

$$\mathbf{W_e} \in \mathbb{R}^{V \times D}$$

When a specific token ID $i$ enters the layer, it is represented mathematically as a **one-hot encoded row vector** $\mathbf{x_i}$ of size $V$, containing all zeros except for a value of $1$ at index $i$:

$$\mathbf{x_i} = [0, 0, \dots, 1, \dots, 0]$$

The embedding extraction operation is a formal matrix multiplication between this one-hot row vector and the embedding weights matrix:

$$\mathbf{e_i} = \mathbf{x_i} \cdot \mathbf{W_e}$$

Because $\mathbf{x_i}$ isolates a single active index, this matrix multiplication simplifies computationally into a direct slice, extracting the **$i$-th row** of $\mathbf{W_e}$.

# <span style="color:#0275D8">Byte-Pair Encoding (BPE) Deep Dive</span>

## <span style="color:#20639B">What is Byte-Pair Encoding (BPE)?</span>
BPE was originally a data compression algorithm later adapted to NLP for subword tokenization. It addresses a critical trade-off:
* **Word-level tokenization** creates massive vocabularies and suffers from Out-Of-Vocabulary (OOV) errors when encountering new words (e.g., training on "smart" but encountering "smartest").
* **Character-level tokenization** keeps the vocabulary tiny but forces the model to process incredibly long sequences, making it harder to learn long-range dependencies.

BPE strikes a balance by starting at the character level and iteratively merging the most frequently adjacent pairs of characters or subwords into new, larger tokens.

## <span style="color:#20639B">Types of BPE</span>
* <span style="color:#3CAEA3">**Standard (Character-Level) BPE:**</span> The text is split into individual characters. Merges are performed strictly based on raw character frequencies.
* <span style="color:#ED553B">**Byte-Level BPE (BBPE):**</span> Used by modern architectures like GPT-2, GPT-3, GPT-4, and LLaMA. Instead of characters, it treats the input string as a sequence of bytes (0 to 255). This allows the tokenizer to handle any language, symbol, or emoji without needing an explicit `[UNK]` (Unknown) token, as every possible text character can ultimately be reduced to basic underlying bytes.

## <span style="color:#20639B">The Math & Logic Behind BPE</span>
The logical core of BPE relies on greedy frequency maximization.

Let the corpus be represented as a sequence of symbols. At any iteration $t$, we compute the joint probability or absolute frequency of all adjacent symbol bigrams $(s_i, s_j)$.

The algorithm selects the optimal pair $(s_i^*, s_j^*)$ that satisfies:
$$(s_i^*, s_j^*) = \arg\max_{(s_i, s_j)} \text{Frequency}(s_i, s_j)$$

Once identified, every instance of the sequence $s_i^* s_j^*$ in the corpus is replaced by a new atomic symbol $s_{\text{new}}$. The base vocabulary is updated:
$$V_{t+1} = V_t \cup \{s_{\text{new}}\}$$

This loop repeats until either the vocabulary reaches a pre-defined maximum target size $|V|$, or the maximum frequency of any remaining pair drops below a specified threshold.

---




# <span style="color:#0275D8">End-to-End Implementation with a Hugging Face Dataset</span>

Below is a complete, production-grade Python script. It pulls a real text dataset from Hugging Face, constructs a custom BPE tokenizer completely from scratch, trains it on the text, and demonstrates how those tokens pass directly into a PyTorch embedding table.

In [1]:
!pip install datasets torch

## small text

In [3]:
text = """
low lower newest widest
low lowest
newer wider
"""

In [4]:
from collections import Counter

words =text.split()
words

['low', 'lower', 'newest', 'widest', 'low', 'lowest', 'newer', 'wider']

In [6]:
vocab = Counter()

for word in words:
    chars = " ".join(list(word)) + " </w>"
    vocab[chars] +=1

print(vocab)

Counter({'l o w </w>': 2, 'l o w e r </w>': 1, 'n e w e s t </w>': 1, 'w i d e s t </w>': 1, 'l o w e s t </w>': 1, 'n e w e r </w>': 1, 'w i d e r </w>': 1})


In [8]:
def get_pair_counts(vocab):
    pairs = Counter()

    for word, freq in vocab.items():
        symbols = word.split()

        for i in range(len(symbols)-1):
            pair = (symbols[i],symbols[i+1])
            pairs[pair] += freq

    return pairs

In [9]:
def merge_pair(pair, vocab):

    new_vocab = {}

    bigram = " ".join(pair)
    replacement = "".join(pair)

    for word in vocab:

        new_word = word.replace(bigram, replacement)

        new_vocab[new_word] = vocab[word]

    return new_vocab

In [10]:
num_merges = 10

for i in range(num_merges):

    pairs = get_pair_counts(vocab)

    if not pairs:
        break

    best_pair = max(pairs, key=pairs.get)

    print(f"Step {i+1}")
    print("Best Pair:", best_pair)

    vocab = merge_pair(best_pair, vocab)

    print(vocab)
    print("-" * 50)

Step 1
Best Pair: ('l', 'o')
{'lo w </w>': 2, 'lo w e r </w>': 1, 'n e w e s t </w>': 1, 'w i d e s t </w>': 1, 'lo w e s t </w>': 1, 'n e w e r </w>': 1, 'w i d e r </w>': 1}
--------------------------------------------------
Step 2
Best Pair: ('lo', 'w')
{'low </w>': 2, 'low e r </w>': 1, 'n e w e s t </w>': 1, 'w i d e s t </w>': 1, 'low e s t </w>': 1, 'n e w e r </w>': 1, 'w i d e r </w>': 1}
--------------------------------------------------
Step 3
Best Pair: ('e', 'r')
{'low </w>': 2, 'low er </w>': 1, 'n e w e s t </w>': 1, 'w i d e s t </w>': 1, 'low e s t </w>': 1, 'n e w er </w>': 1, 'w i d er </w>': 1}
--------------------------------------------------
Step 4
Best Pair: ('er', '</w>')
{'low </w>': 2, 'low er</w>': 1, 'n e w e s t </w>': 1, 'w i d e s t </w>': 1, 'low e s t </w>': 1, 'n e w er</w>': 1, 'w i d er</w>': 1}
--------------------------------------------------
Step 5
Best Pair: ('e', 's')
{'low </w>': 2, 'low er</w>': 1, 'n e w es t </w>': 1, 'w i d es t </w>': 1,

In [17]:
vocab = {
    "low": 0,
    "er": 1,
    "new": 2,
    "est": 3
}

In [18]:
import numpy as np

embedding_matrix = np.random.randn(4,5)
print(embedding_matrix)

[[-0.70629605  0.71276706 -0.23594908 -0.69464039  1.50885842]
 [-0.05211041 -0.25328265  0.0812527   0.33929924  1.37588786]
 [ 0.6326397  -1.05184077  1.3573328   0.8434889   1.05611394]
 [ 0.45370927 -0.88910788 -1.18547711  0.41731202 -0.58538209]]


In [19]:
tokens = ["low", "er"]

In [20]:
token_ids = [vocab[t] for t in tokens]

print(token_ids)

[0, 1]


In [21]:
embeddings = embedding_matrix[token_ids]

In [22]:
embeddings

array([[-0.70629605,  0.71276706, -0.23594908, -0.69464039,  1.50885842],
       [-0.05211041, -0.25328265,  0.0812527 ,  0.33929924,  1.37588786]])

In [25]:
import torch
import torch.nn as nn

embedding = nn.Embedding(
    num_embeddings=4,
    embedding_dim=5
)

In [26]:
input_ids = torch.tensor([0,1])

In [28]:
output = embedding(input_ids)

In [29]:
output

tensor([[ 0.5238,  1.3712, -0.1714, -0.3869, -1.5427],
        [ 0.0924, -0.1399, -1.1936,  2.6482,  1.0012]],
       grad_fn=<EmbeddingBackward0>)

In [31]:
output.shape

torch.Size([2, 5])

In [32]:
output = embedding_matrix[input_ids]
output

array([[-0.70629605,  0.71276706, -0.23594908, -0.69464039,  1.50885842],
       [-0.05211041, -0.25328265,  0.0812527 ,  0.33929924,  1.37588786]])

In [33]:
import torch
import torch.nn as nn

# Vocabulary
vocab = {
    "low": 0,
    "lower": 1,
    "new": 2,
    "wide": 3
}

id_to_word = {v:k for k,v in vocab.items()}

# Embedding Layer
embedding = nn.Embedding(
    num_embeddings=4,
    embedding_dim=5
)

# Get embedding for "low"
word = "low"

word_id = vocab[word]

vector = embedding(torch.tensor(word_id))

print("Word :", word)
print("ID   :", word_id)
print("Vector:")
print(vector)

Word : low
ID   : 0
Vector:
tensor([-0.5804,  0.8109, -0.3290,  0.7004,  0.2989],
       grad_fn=<EmbeddingBackward0>)


## BPE

In [34]:
import collections
from datasets import load_dataset
import torch
import torch.nn as nn

dataset = load_dataset("ag_news",split="train[:500]")
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 500
})

In [35]:
dataset['text']

Column(["Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.', "Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums.", 'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.', 'Oil prices soar to all-time record, posing new menace to US economy

In [36]:
dataset['label']

Column([2, 2, 2, 2, 2, ...])

In [37]:
corpus = dataset['text']
print(len(corpus))

500


In [38]:
class ScratchBPETokenizer:
    def __init__(self, target_vocab_size):
        self.target_vocab_size = target_vocab_size
        self.vocab = {}
        self.merges = {}
        self.special_token = "</w>"

    def _get_stats(self, splits, word_counts):
        """Counts frequencies of adjacent pairs."""
        pairs = collections.defaultdict(int)
        for word, freq in word_counts.items():
            symbols = splits[word]
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return pairs

    def _merge_vocab(self, pair, splits):
        """Merges the chosen pair across the split dictionary representation."""
        new_splits = {}
        bigram = list(pair)
        for word in splits:
            w = splits[word]
            i = 0
            new_w = []
            while i < len(w):
                if i < len(w) - 1 and w[i] == bigram[0] and w[i+1] == bigram[1]:
                    new_w.append(bigram[0] + bigram[1])
                    i += 2
                else:
                    new_w.append(w[i])
                    i += 1
            new_splits[word] = new_w
        return new_splits

    def train(self, corpus):
        print("\n--- Training BPE Tokenizer ---")
        # Step A: Compute base word frequencies from raw corpus text
        word_counts = collections.defaultdict(int)
        for text in corpus:
            # Simple whitespace cleanup and token split
            for word in text.strip().split():
                word_counts[word] += 1

        # Step B: Initialize splits by separating characters and appending end-of-word marker
        splits = {word: [char for char in word] + [self.special_token] for word in word_counts}
        
        # Step C: Establish base alphabet vocabulary
        base_vocab = set()
        for word in word_counts:
            for char in word:
                base_vocab.add(char)
        base_vocab.add(self.special_token)
        
        # Current vocabulary is represented as a list matching indices to strings
        current_vocab = list(base_vocab)
        num_merges = self.target_vocab_size - len(current_vocab)
        
        print(f"Base Alphabet Size: {len(current_vocab)}")
        print(f"Targeting {num_merges} merges to reach Vocab Size: {self.target_vocab_size}")

        # Step D: Main iteration loop for token creation
        for iteration in range(num_merges):
            pairs = self._get_stats(splits, word_counts)
            if not pairs:
                print("No more pairs left to merge.")
                break
                
            # Find the most frequent pair
            best_pair = max(pairs, key=pairs.get)
            new_token = best_pair[0] + best_pair[1]
            
            # Record the merge rule and apply it
            self.merges[best_pair] = new_token
            splits = self._merge_vocab(best_pair, splits)
            current_vocab.append(new_token)
            
            if (iteration + 1) % 50 == 0 or (iteration + 1) == num_merges:
                print(f"Merge {iteration + 1}/{num_merges}: {best_pair} -> '{new_token}'")

        # Create final mapping tables
        self.token_to_id = {token: idx for idx, token in enumerate(current_vocab)}
        self.id_to_token = {idx: token for idx, token in enumerate(current_vocab)}
        print("Tokenizer training complete!")

    def tokenize(self, text):
        """Tokenizes raw text strings down into subword token segments using learned rules."""
        words = text.strip().split()
        final_tokens = []
        
        for word in words:
            # Start with characters
            word_splits = [char for char in word] + [self.special_token]
            
            # Iteratively apply learned merge rules in the exact sequence they were discovered
            for pair, new_token in self.merges.items():
                i = 0
                new_splits = []
                while i < len(word_splits):
                    if i < len(word_splits) - 1 and word_splits[i] == pair[0] and word_splits[i+1] == pair[1]:
                        new_splits.append(new_token)
                        i += 2
                    else:
                        new_splits.append(word_splits[i])
                        i += 1
                word_splits = new_splits
            
            final_tokens.extend(word_splits)
            
        # Convert tokens to their registered structural Integer IDs
        # Default to index 0 if a token character wasn't seen in training text
        ids = [self.token_to_id.get(token, 0) for token in final_tokens]
        return final_tokens, ids

In [39]:
TARGET_VOCAB_SIZE = 300
tokenizer = ScratchBPETokenizer(target_vocab_size=TARGET_VOCAB_SIZE)
tokenizer.train(corpus)


--- Training BPE Tokenizer ---
Base Alphabet Size: 82
Targeting 218 merges to reach Vocab Size: 300
Merge 50/218: ('l', 'o') -> 'lo'
Merge 100/218: ('s', 'p') -> 'sp'
Merge 150/218: ('t', 's</w>') -> 'ts</w>'
Merge 200/218: ('s', 't</w>') -> 'st</w>'
Merge 218/218: ('p', 'or') -> 'por'
Tokenizer training complete!


In [40]:
sample_sentence = "The updates showing market trends are spectacular."
tokens, token_ids = tokenizer.tokenize(sample_sentence)

print("\n--- Tokenization Output Example ---")
print(f"Input Sentence: '{sample_sentence}'")
print(f"Generated Tokens: {tokens}")
print(f"Generated Token IDs: {token_ids}")


--- Tokenization Output Example ---
Input Sentence: 'The updates showing market trends are spectacular.'
Generated Tokens: ['The</w>', 'u', 'p', 'd', 'at', 'es</w>', 'sh', 'ow', 'ing</w>', 'm', 'ar', 'k', 'et</w>', 'tr', 'en', 'd', 's</w>', 'are</w>', 'sp', 'ec', 't', 'ac', 'ul', 'ar', '.</w>']
Generated Token IDs: [186, 60, 0, 14, 116, 115, 192, 151, 102, 15, 93, 3, 200, 228, 95, 14, 83, 176, 181, 125, 63, 118, 206, 93, 97]


In [42]:
EMBEDDING_DIM = 8 

In [43]:
embedding_layer = nn.Embedding(num_embeddings=TARGET_VOCAB_SIZE, embedding_dim=EMBEDDING_DIM)

In [44]:
embedding_layer

Embedding(300, 8)

In [45]:
input_tensor = torch.tensor(token_ids, dtype=torch.long)

In [46]:
input_tensor

tensor([186,  60,   0,  14, 116, 115, 192, 151, 102,  15,  93,   3, 200, 228,
         95,  14,  83, 176, 181, 125,  63, 118, 206,  93,  97])

In [47]:
with torch.no_grad():
    dense_vectors = embedding_layer(input_tensor)

print("\n--- Continuous Embedding Conversion ---")
print(f"Output Matrix Shape (Sequence Length x Embedding Dim): {dense_vectors.shape}")
print("First token vector representation:\n", dense_vectors[0].numpy())


--- Continuous Embedding Conversion ---
Output Matrix Shape (Sequence Length x Embedding Dim): torch.Size([25, 8])
First token vector representation:
 [-0.73676306 -1.6620272   0.09096897  1.174728   -1.37724     0.5829098
  0.19594815  0.9717319 ]


In [48]:
import torch.nn.functional as F


vector_1 = dense_vectors[0].unsqueeze(0) # Shape: [1, 8]
vector_2 = dense_vectors[1].unsqueeze(0) # Shape: [1, 8]


similarity = F.cosine_similarity(vector_1, vector_2)
print(f"Cosine Similarity between Token 1 and Token 2: {similarity.item():.4f}")

Cosine Similarity between Token 1 and Token 2: 0.0901
